In [32]:
import numpy as np
import matplotlib.pyplot as plt

def lin_interpolate(nodes):
    nodes = np.array(sorted(nodes, key=lambda x: x[0]))
    nodes_x = [node[0] for node in nodes]
    nodes_y = [node[1] for node in nodes]

    def lin_int(x):
        # if x outside data range return None;
        # if x already in data range, return known y value
        if x < nodes_x[0] or x > nodes_x[-1]: return None;
        if x in nodes_x:
            return nodes_y[nodes_x.index(x)];

        else:
            # gets i : x \in [a_{i-1}, a_i)
            i = np.searchsorted(nodes_x, x, side="right")
            x0, x1 = nodes_x[i-1], nodes_x[i]
            y0, y1 = nodes_y[i-1], nodes_y[i]
            return y0*(x1-x)/(x1-x0) + y1*(x-x0)/(x1-x0);

    return np.vectorize(lin_int)

def log_interpolate(nodes):
    nodes = np.array(sorted(nodes, key=lambda x: x[0]))
    nodes_x = [node[0] for node in nodes]
    nodes_y = [node[1] for node in nodes]


def R_curve(par_yields, frequency):
    nodes_tau, nodes_R = zip(*par_yields)

    tt = np.linspace(nodes_tau[0], nodes_tau[-1], frequency * nodes_tau[-1])
    R_interpl = lin_interpolate(par_yields)
    RR = R_interpl(tt)

    return list(zip(tt, RR))


def D_bootstrap(R_t):
    # coupon_pars should be a list of (t,ρ_t)
    taus, Rvals = zip(*R_t)
    rhos = [R/2 for R in Rvals]

    rho_1 = rhos[0]
    d_factors = [1/(1+rho_1)]

    for i in range(1, len(rhos)):
        rho_i = rhos[i]  # technically this is rho_{t_{i+1}}, but no matter
        d_i = (1-rho_i*sum(d_factors[:i]))/(1+rho_i);
        d_factors.append(d_i)

    return list(zip(taus, d_factors))


def D_to_Zc(D_t):
    # convert discounts to annually continuous compounding zero rates
    return [(t, -(1/t)*np.log(d_t)) for t, d_t in D_t]

def D_to_Zm(D_t, freq):
    # convert discounts to m-times-annually compounding zero rates
    m = freq;
    return [(t, m*(d_t**(-1/(m*t))-1)) for t, d_t in D_t]

def Zc_to_Zm(Zc_t, freq):
    # converts annual continuous zero-rates into m-times-annually compounding
    m = freq;
    return [(t, m*(np.exp(zc_t/m)-1)) for t, zc_t in Zc_t]

def Zm_to_Zc(Zc_t, freq):
    # converts m-times-annually compounding rates into annual continuous zero-rates
    m = freq;
    return [(t, m*np.log(1+zm_t/m)) for t, zm_t in Zm_t]


# ===============================================

def main():
    pars=[
        # (0.125, 4.7),
        # (0.25, 4.63),
        (0.5, 4.44),
        (1.0, 4.33),
        (2.0, 4.29),
        (3.0, 4.25),
        (5.0, 4.28),
        (7.0, 4.35),
        (10.0, 4.42),
        (20, 4.7),
        (30, 4.61)
    ]
    pars = [(tau, R/100) for tau,R in pars]
    nodes_t = [t for t, r in pars]

    freq = 2

    R_t = R_curve(pars, freq)
    D_t = D_bootstrap(R_t)
    Zc_t = D_to_Zc(D_t)
    Zm_t = D_to_Zm(D_t, 12)
    Z12_t = Zc_to_Zm(Zc_t, 12)


    print("tau,\t R_t,\t\t D_t,\t\t\t Z_t")
    print("=====================================")
    for i in range(len(D_t)):
        r = R_t[i]
        d = D_t[i]
        z = Z12_t[i]

        if r[0] in [0.5,1.0,2.0,3.0,5.0] :
            print(f"{r[0]:.3f},\t {r[1]:.4f},\t {d[1]:.7f},\t {z[1]/12:.5%}")
    print()

    # ===========================================
    # Plotting
    # xx, yy = zip(*R_t)
    #
    # xx=[12*x for x in xx] # tau -> months
    # yy=[100*y for y in yy] # percentage

    # fig, ax = plt.subplots(1,1,figsize=plt.figaspect(0.5))
    # ax.plot(xx, yy)
    # ax.set_ylim(4.10, 4.80)
    # ax.minorticks_on()
    # ax.grid(axis="y")
    # plt.show()




main()

tau,	 R_t,		 D_t,			 Z_t
0.500,	 0.0444,	 0.9782821,	 0.36662%
1.000,	 0.0433,	 0.9580778,	 0.35752%
2.000,	 0.0429,	 0.9186390,	 0.35422%
3.000,	 0.0425,	 0.8815439,	 0.35084%
5.000,	 0.0428,	 0.8091546,	 0.35357%



0.5, 0.3665017517%,
1.0, 0.3573693182%,
2.0, 0.3539768643%,
3.0, 0.3505464124%
5.0, 0.3529873350%

In [45]:
d = 0.9381868
tau = 1.5

zc = -(1/tau)*np.log(d)
z12 = (d**(-1/(12*tau))-1)

In [46]:
print(f"monthly: {z12:.5%}, cont: {zc:.5%}")

monthly: 0.35511%, cont: 4.25375%


In [ ]:
https://home.treasury.gov/resource-center/data-chart-center/interest-rates/pages/xml?data=daily_treasury_yield_curve&field_tdr_date_value_month=202204

In [ ]:
import json, time
import typing
import logging, requests

from pathlib import Path
from dataclasses import dataclass
from urllib.parse import urlencode

# Useful 'constants'
SEARCH_BASE_URL = "https://replay.pokemonshowdown.com/api/replays/search"
REPLAY_BASE_URL = "https://replay.pokemonshowdown.com/"
USER_BASE_URL = "https://pokemonshowdown.com/users/"

# Only needed if you want messages to appear when an error, etc. occurs
logger = logging.getLogger()
logger.setLevel(logging.ERROR)

    params = urlencode({"format": fmt, "page": str(page_num)})
    url = f"{SEARCH_BASE_URL}?{params}"

    # This deals with an apparent quirk in PS's search: the first-page results are fine,
    # but subsequent searches seem to want this `Referer` in the GET request.
    headers = {}
    if page_num > 1:
        headers = {"Referer": f"https://replay.pokemonshowdown.com/?format={fmt}&page={page_num}"}

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.RequestException as e:
        logger.error("failed to get page %d: %s", page_num, e)
        return []

    # The API response starts with a leading character ']' before valid JSON,
    # so remove it
    raw = response.content[1:]

    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        logger.error("failed to parse page %d: %s", page_num, e)
        return []
    return data


